# Ecommerce OPK PydanticAI Migration Process

## Initialization

In [141]:
import os
import asyncio
import concurrent.futures
from datetime import datetime
from httpx import AsyncClient
from dotenv import load_dotenv
from pydantic import BaseModel, create_model
from pydantic_ai import Agent, RunContext
from dataclasses import dataclass
from typing import List, Optional, Dict, Any, Awaitable, Type
from abc import ABC, abstractmethod
from enum import Enum
from wireup import create_container, abstract, container, service

load_dotenv()

/tmp/ipykernel_146212/1925801720.py:13: DeprecationWarning: Using the wireup.container singleton is deprecated. Create your own instance of the container using wireup.create_container. See: https://maldoinc.github.io/wireup/latest/getting_started/
  from wireup import create_container, abstract, container, service


True

Defining the class models

In [142]:
class ProductCategory(BaseModel):
	id: int
	name: str
	description: str
	product_count: int
	created_at: datetime
	updated_at: Optional[datetime] = None
	products: Optional[List["Product"]] = None

	class Config:
		arbitrary_types_allowed = True

class Product(BaseModel):
	id: int
	name: str
	description: str
	product_variant_count: int
	product_category_id: int
	created_at: datetime
	updated_at: Optional[datetime] = None
	product_variants: Optional[List["ProductVariant"]] = None
	product_category: Optional["ProductCategory"] = None

	class Config:
		arbitrary_types_allowed = True

class ProductVariant(BaseModel):
	id: int
	name: str
	price: float
	stock: int
	product_id: int
	created_at: datetime
	updated_at: Optional[datetime] = None
	product: Optional["Product"] = None
	cart_items: Optional[List["CartItem"]] = None
	order_items: Optional[List["OrderItem"]] = None

	class Config:
		arbitrary_types_allowed = True

class User(BaseModel):
	id: int
	name: str
	email: str
	phone_number: str
	address_count: int
	cart_item_count: int
	created_at: datetime
	updated_at: Optional[datetime] = None
	addresses: Optional[List["Address"]] = None
	cart_items: Optional[List["CartItem"]] = None
	orders: Optional[List["Order"]] = None

	class Config:
		arbitrary_types_allowed = True

class Address(BaseModel):
	id: int
	user_id: int
	address_line: str
	city: str
	state: str
	postal_code: str
	country: str
	created_at: datetime
	updated_at: Optional[datetime] = None
	user: Optional["User"] = None
	orders: Optional[List["Order"]] = None

	class Config:
		arbitrary_types_allowed = True

class CartItem(BaseModel):
	id: int
	quantity: int
	user_id: int
	product_variant_id: int
	created_at: datetime
	updated_at: Optional[datetime] = None
	user: Optional["User"] = None
	product_variant: Optional["ProductVariant"] = None

	class Config:
		arbitrary_types_allowed = True

class OrderItem(BaseModel):
	id: int
	order_id: int
	product_variant_id: int
	quantity: int
	total_price: float
	created_at: datetime
	updated_at: Optional[datetime] = None
	order: Optional["Order"] = None
	product_variant: Optional["ProductVariant"] = None

	class Config:
		arbitrary_types_allowed = True

class OrderStatus(Enum):
	PENDING = "PENDING"
	TO_PAY = "TO_PAY"
	TO_SHIP = "TO_SHIP"
	TO_RECEIVE = "TO_RECEIVE"
	COMPLETED = "COMPLETED"
	DELIVERED = "DELIVERED"

class Order(BaseModel):
	id: int
	user_id: int
	address_id: int
	status: OrderStatus
	total_price: float
	created_at: datetime
	updated_at: Optional[datetime] = None
	user: Optional["User"] = None
	address: Optional["Address"] = None
	order_items: Optional[List["OrderItem"]] = None

	class Config:
		use_enum_values = True
		arbitrary_types_allowed = True



## Execution

### API HTTP Client

Creating an API HTTP client interface

In [143]:
class FetchProductsByProductCategoryParams(BaseModel):
	product_category_id: int

class FetchProductParams(BaseModel):
	product_id: int

class FetchProductVariantParams(BaseModel):
	product_variant_id: int

class FetchProductVariantsByProductParams(BaseModel):
	product_id: int

class FetchUserByEmailParams(BaseModel):
	email: str

class FetchUserParams(BaseModel):
	user_id: int

class FetchAddressesByUserParams(BaseModel):
	user_id: int

class CreateAddressParams(BaseModel):
	user_id: int
	address_line: str
	city: str
	state: str
	postal_code: str	
	country: str

class FetchAddressParams(BaseModel):
	address_id: int

class FetchCartItemsByUserParams(BaseModel):
	user_id: int

class CreateCartItemParams(BaseModel):
	user_id: int
	product_variant_id: int
	quantity: int

class FetchCartItemParams(BaseModel):
	cart_item_id: int

class DeleteCartItemParams(BaseModel):
	cart_item_id: int

class FetchOrderItemParams(BaseModel):
	order_item_id: int

class CreateOrderItemParams(BaseModel):
	order_id: int
	product_variant_id: int
	quantity: int

class FetchOrderParams(BaseModel):
	order_id: int

class CreateOrderParams(BaseModel):
	user_id: int
	address_id: int
	total_price: float

class UpdateOrderParams(BaseModel):
	order_id: int
	status: OrderStatus
	total_price: float

	class Config:
		use_enum_values = True

In [144]:
@container.abstract
class CoreApiHttpClientABC(ABC):
	@abstractmethod
	async def fetch_product_categories(self) -> List[ProductCategory]:
		pass

	@abstractmethod
	async def fetch_products_by_product_category(self, params: FetchProductsByProductCategoryParams) -> List[Product]:
		pass

	@abstractmethod
	async def fetch_product(self, params: FetchProductParams) -> Product:
		pass

	@abstractmethod
	async def fetch_product_variant(self, params: FetchProductVariantParams) -> ProductVariant:
		pass

	@abstractmethod
	async def fetch_product_variants_by_product(self, params: FetchProductVariantsByProductParams) -> List[ProductVariant]:
		pass

	@abstractmethod
	async def fetch_user(self, params: FetchUserParams) -> User:
		pass

	@abstractmethod
	async def fetch_user_by_email(self, params: FetchUserByEmailParams) -> Optional[User]:
		pass

	@abstractmethod
	async def fetch_addresses_by_user(self, params: FetchAddressesByUserParams) -> List[Address]:
		pass

	@abstractmethod
	async def create_address(self, params: CreateAddressParams) -> Address:
		pass

	@abstractmethod
	async def fetch_address(self, params: FetchAddressParams) -> Address:
		pass

	@abstractmethod
	async def fetch_cart_items_by_user(self, params: FetchCartItemsByUserParams) -> List[CartItem]:
		pass

	@abstractmethod
	async def fetch_cart_item(self, params: FetchCartItemParams) -> CartItem:
		pass

	@abstractmethod
	async def create_cart_item(self, params: CreateCartItemParams) -> CartItem:
		pass

	@abstractmethod
	async def delete_cart_item(self, params: DeleteCartItemParams) -> CartItem:
		pass

	@abstractmethod
	async def fetch_order_item(self, params: CreateCartItemParams) -> CartItem:
		pass

	@abstractmethod
	async def create_order_item(self, params: CreateOrderItemParams) -> CartItem:
		pass

	@abstractmethod
	async def fetch_order(self, params: FetchOrderParams) -> Order:
		pass

	@abstractmethod
	async def create_order(self, params: CreateOrderParams) -> Order:
		pass

	@abstractmethod
	async def update_order(self, params: UpdateOrderParams) -> Order:
		pass

API HTTP client concrete implementatoin

In [145]:
class CoreApiHttpClientConfig:
	base_url: str = os.getenv("NOCODB_API_BASE_URL")
	access_token: str = os.getenv("NOCODB_XC_TOKEN")
	headers: Dict[str, str] = {"accept": "application/json", "xc-token": access_token}

	table_mappings = dict(
		users = dict(
			id = "mtkw0ob3dxznnoy",
			relationships = dict(
				addresses = "caph3486tt4sx7o",
				orders = "cdfama7at5q7m06",
				cart_items = "crsqehtu2hhjmr4",
				tickets = "cc6k8jxk403h9vw"
			)
		),
		addresses = dict(
			id = "mazzr6vdh8rnwqa",
			relationships = dict(
				users = "cw5d6yvxye01jtt",
				orders = "ct1wu3stl9htv3g",
			)
		),
		products = dict(
			id = "m9z4v3gwund7k0y",
			relationships = dict(
				product_variants = "cnwqa1j9ql3ihd0",
				product_categories = "c30k59j0ga34qes",	
			)
		),
		cart_items = dict(
			id = "mc8v3m1u6qvvgel",
			relationships = dict(
				users = "c7x5dlf7fi4d188",
				product_variants = "cywylwil02ezy57"
			)
		),
		orders = dict(
			id = "mohiyayf8xxvffu",
			relationships = dict(
				users = "ctuyhgcixcqx4jx",	
				addresses = "cbz40wlxtlkmrfb",
				orders_items = "cy92jvur8q9slyp"
			)
		),
		product_variants = dict(
			id = "mjbuzjm1nf6jfd5",	
			relationships = dict(
				products = "cduhu2zr2cgm50j",
				cart_items = "ci2poafzpi7hc4v",
				order_items = "c58f9xst5r1n4zv"
			)
		),
		order_items = dict(
			id = "m50bskuo8h6kpmn",
			relationships = dict(
				orders = "c454r9wttyif0pz",
				product_variants = "c9r4mt54o5p26tm"
			)
		),
		product_categories = dict(
			id = "m8myg8hob1pyii0",
			relationships = dict(
				products = "cbtac8lok4eeekr"
			)
		),
		tickets = dict(
			id = "m5zmlzo25vtcxhc",	
			relationships = dict(
				users = "cxb7xm08171uu9k",
			)
		)
	)

In [146]:
@container.register
class CoreApiHttpClient(CoreApiHttpClientABC):
	def __init__(self):
		base_url: str = CoreApiHttpClientConfig.base_url
		access_token: str = CoreApiHttpClientConfig.access_token
		headers: Dict[str, str] = {"accept": "application/json", "xc-token": access_token}

		self._http_client = AsyncClient(base_url=base_url, headers=headers)

	async def fetch_product_categories(self) -> List[ProductCategory]:
		product_categories_table_id = CoreApiHttpClientConfig.table_mappings['product_categories']['id']
		end_point = f"/api/v2/tables/{product_categories_table_id}/records"
		response = await self._http_client.get(end_point)
		response.raise_for_status()

		# Get the list of product categories
		unparsed_product_categories = response.json().get('list')

		# Parse each ProductCategory object into a ProductCategory object
		def parse_product_category(unparsed_product_category: Dict[str, Any]) -> ProductCategory:
			return ProductCategory(
				id=unparsed_product_category.get('Id'),
				name=unparsed_product_category.get('Name'),
				description=unparsed_product_category.get('Description'),
				product_count=unparsed_product_category.get('Products'),
				created_at=unparsed_product_category.get('CreatedAt'), 
				updated_at=unparsed_product_category.get('UpdatedAt'),	
			)

		# Use a ThreadPoolExecutor to parallelize the parsing of the ProductCategory objects
		with concurrent.futures.ThreadPoolExecutor() as executor:
			parsed_product_categories = list(executor.map(parse_product_category, unparsed_product_categories))

		return parsed_product_categories

	async def fetch_products_by_product_category(self, params: FetchProductsByProductCategoryParams) -> List[Product]:
		product_categories_table_id = CoreApiHttpClientConfig.table_mappings['product_categories']['id']
		products_relationship_id = CoreApiHttpClientConfig.table_mappings['product_categories']['relationships']['products']

		include_fields = ["Id", "Title", "Name", "Description", "ProductCategories_id", "ProductVariants", "CreatedAt", "UpdatedAt"]
		end_point = f"/api/v2/tables/{product_categories_table_id}/links/{products_relationship_id}/records/{params.product_category_id}?fields={','.join(include_fields)}"
		response = await self._http_client.get(end_point)
		response.raise_for_status()
		
		# Get the list of products
		unparsed_products = response.json().get('list')

		# Parse each Product object into a Product object
		def parse_product(unparsed_product: Dict[str, Any]) -> Product:
			return Product(
				id=unparsed_product.get('Id'),
				name=unparsed_product.get('Name'),
				description=unparsed_product.get('Description'),
				product_variant_count=unparsed_product.get('ProductVariants'),
				product_category_id=unparsed_product.get('ProductCategories_id'),
				created_at=unparsed_product.get('CreatedAt'), 
				updated_at=unparsed_product.get('UpdatedAt'),	
			)

		# Use a ThreadPoolExecutor to parallelize the parsing of the Product objects
		with concurrent.futures.ThreadPoolExecutor() as executor:
			parsed_products = list(executor.map(parse_product, unparsed_products))

		return parsed_products

	async def fetch_product(self, params: FetchProductParams) -> Product:
		products_table_id = CoreApiHttpClientConfig.table_mappings['products']['id']
		end_point = f"/api/v2/tables/{products_table_id}/records/{params.product_id}"
		response = await self._http_client.get(end_point)
		response.raise_for_status()

		# Parse the Product object into a Product object
		def parse_product(unparsed_product: Dict[str, Any]) -> Product:
			return Product(
				id=unparsed_product.get('Id'),
				name=unparsed_product.get('Name'),
				description=unparsed_product.get('Description'),
				product_variant_count=unparsed_product.get('ProductVariants'),
				product_category_id=unparsed_product.get('ProductCategories_id'),
				created_at=unparsed_product.get('CreatedAt'), 
				updated_at=unparsed_product.get('UpdatedAt'),	
			)

		return parse_product(response.json())

	async def fetch_product_variant(self, params: FetchProductVariantParams) -> ProductVariant:
		product_variants_table_id = CoreApiHttpClientConfig.table_mappings['product_variants']['id']
		end_point = f"/api/v2/tables/{product_variants_table_id}/records/{params.product_variant_id}"
		response = await self._http_client.get(end_point)
		response.raise_for_status()

		# Parse the ProductVariant object into a ProductVariant object
		def parse_product_variant(unparsed_product_variant: Dict[str, Any]) -> ProductVariant:
			return ProductVariant(
				id=unparsed_product_variant.get('Id'),
				name=unparsed_product_variant.get('Name'),
				price=unparsed_product_variant.get('Price'),
				stock=unparsed_product_variant.get('Stock'),
				product_id=unparsed_product_variant.get('Products_id'),
				created_at=unparsed_product_variant.get('CreatedAt'), 
				updated_at=unparsed_product_variant.get('UpdatedAt'),	
			)

		product_variant = parse_product_variant(response.json())
		return product_variant

	async def fetch_product_variants_by_product(self, params: FetchProductVariantsByProductParams) -> List[ProductVariant]:
		products_table_id = CoreApiHttpClientConfig.table_mappings['products']['id']
		product_variants_relationship_id = CoreApiHttpClientConfig.table_mappings['products']['relationships']['product_variants']
		include_fields = ["Id", "Title", "Name", "Description", "Price", "Stock", "Products_id", "CreatedAt", "UpdatedAt"]
		end_point = f"/api/v2/tables/{products_table_id}/links/{product_variants_relationship_id}/records/{params.product_id}?fields={','.join(include_fields)}"
		response = await self._http_client.get(end_point)
		response.raise_for_status()

		# Get the list of product variants
		unparsed_product_variants = response.json().get('list')

		# Parse each ProductVariant object into a ProductVariant object
		def parse_product_variant(unparsed_product_variant: Dict[str, Any]) -> ProductVariant:
			return ProductVariant(
				id=unparsed_product_variant.get('Id'),
				name=unparsed_product_variant.get('Name'),
				price=unparsed_product_variant.get('Price'),
				stock=unparsed_product_variant.get('Stock'),
				product_id=unparsed_product_variant.get('Products_id'),
				created_at=unparsed_product_variant.get('CreatedAt'), 
				updated_at=unparsed_product_variant.get('UpdatedAt'),	
			)

		# Use a ThreadPoolExecutor to parallelize the parsing of the ProductVariant objects
		with concurrent.futures.ThreadPoolExecutor() as executor:
			product_variants = list(executor.map(parse_product_variant, unparsed_product_variants))

		return product_variants

	async def fetch_user(self, params: FetchUserParams) -> User:
		users_table_id = CoreApiHttpClientConfig.table_mappings['users']['id']
		end_point = f"/api/v2/tables/{users_table_id}/records/{params.user_id}"
		response = await self._http_client.get(end_point)
		response.raise_for_status()

		unparsed_user = response.json()
		
		# Parse the User object into a User object
		def parse_user(unparsed_user: Dict[str, Any]) -> User:
			return User(
				id=unparsed_user.get('Id'),	
				name=unparsed_user.get('Name'),
				email=unparsed_user.get('Email'),
				phone_number=unparsed_user.get('PhoneNumber'),
				address_count=unparsed_user.get('Addresses'),
				cart_item_count=unparsed_user.get('CartItems'),
				created_at=unparsed_user.get('CreatedAt'), 
				updated_at=unparsed_user.get('UpdatedAt'),	
			)

		user = parse_user(unparsed_user)
		return user	

	async def fetch_user_by_email(self, params: FetchUserByEmailParams) -> Optional[User]:
		users_table_id = CoreApiHttpClientConfig.table_mappings['users']['id']
		end_point = f"/api/v2/tables/{users_table_id}/records?where=(Email,eq,{params.email})&limit=1" 
		response = await self._http_client.get(end_point)
		response.raise_for_status()

		# Get the list of user
		unparsed_users = response.json().get('list')

		# Parse user into a User object
		def parse_user(unparsed_user: Dict[str, Any]) -> User:
			return User(
				id=unparsed_user.get('Id'),
				name=unparsed_user.get('Name'),
				email=unparsed_user.get('Email'),
				phone_number=unparsed_user.get('PhoneNumber'),
				address_count=unparsed_user.get('Addresses'),
				cart_item_count=unparsed_user.get('CartItems'),
				created_at=unparsed_user.get('CreatedAt'), 
				updated_at=unparsed_user.get('UpdatedAt'),	
			)

		# Use a ThreadPoolExecutor to parallelize the parsing of the User objects
		with concurrent.futures.ThreadPoolExecutor() as executor:
			users = list(executor.map(parse_user, unparsed_users))

		user = users[0] if users else None
		return user

	async def fetch_addresses_by_user(self, params: FetchAddressesByUserParams) -> List[Address]:
		users_table_id = CoreApiHttpClientConfig.table_mappings['users']['id']
		address_relationship_id = CoreApiHttpClientConfig.table_mappings['users']['relationships']['addresses']
		include_fields = ["Id", "Users_id", "AddressLine", "City", "State", "PostalCode", "Country", "CreatedAt", "UpdatedAt"]
		end_point = f"/api/v2/tables/{users_table_id}/links/{address_relationship_id}/records/{params.user_id}?fields={",".join(include_fields)}" 
		response = await self._http_client.get(end_point)
		response.raise_for_status()

		# Get the list of address
		unparsed_addresses = response.json().get('list')

		# Parse address into a Address object
		def parse_address(unparsed_address: Dict[str, Any]) -> Address:
			return Address(
				id=unparsed_address.get('Id'),
				user_id=unparsed_address.get('Users_id'),
				address_line=unparsed_address.get('AddressLine'),
				city=unparsed_address.get('City'),
				state=unparsed_address.get('State'),
				country=unparsed_address.get('Country'),
				postal_code=unparsed_address.get('PostalCode'),
				created_at=unparsed_address.get('CreatedAt'), 
				updated_at=unparsed_address.get('UpdatedAt'),	
			)

		# Use a ThreadPoolExecutor to parallelize the parsing of the Address objects
		with concurrent.futures.ThreadPoolExecutor() as executor:
			addresses = list(executor.map(parse_address, unparsed_addresses))

		return addresses
	
	async def fetch_cart_items_by_user(self, params: FetchCartItemsByUserParams) -> List[CartItem]: 
		users_table_id = CoreApiHttpClientConfig.table_mappings['users']['id']
		cart_items_relationship_id = CoreApiHttpClientConfig.table_mappings['users']['relationships']['cart_items']
		include_fields = ["Id", "Users_id", "ProductVariants_id", "Quantity", "CreatedAt", "UpdatedAt"] 
		end_point = f"/api/v2/tables/{users_table_id}/links/{cart_items_relationship_id}/records/{params.user_id}?fields={",".join(include_fields)}" 
		response = await self._http_client.get(end_point)
		response.raise_for_status()

		# Get the list of cart_item
		unparsed_cart_items = response.json().get('list')

		# Parse cart_item into a Cart_item object
		def parse_cart_item(unparsed_cart_item: Dict[str, Any]) -> CartItem:
			return CartItem(
				id=unparsed_cart_item.get('Id'),
				user_id=unparsed_cart_item.get('Users_id'),
				product_variant_id=unparsed_cart_item.get('ProductVariants_id'),
				quantity=unparsed_cart_item.get('Quantity'),
				created_at=unparsed_cart_item.get('CreatedAt'), 
				updated_at=unparsed_cart_item.get('UpdatedAt'),	
			)

		# Use a ThreadPoolExecutor to parallelize the parsing of the Cart_item objects
		with concurrent.futures.ThreadPoolExecutor() as executor:
			cart_items = list(executor.map(parse_cart_item, unparsed_cart_items))

		return cart_items
	
	async def delete_cart_item(self, params: DeleteCartItemParams) -> CartItem:
		cart_items_table_id = CoreApiHttpClientConfig.table_mappings['cart_items']['id']

		cart_item_to_delete = await self.fetch_cart_item(
			params=FetchCartItemParams(cart_item_id=params.cart_item_id)
		)

		end_point = f"/api/v2/tables/{cart_items_table_id}/records"
		response = await self._http_client.request("DELETE", end_point, json={
			"Id": params.cart_item_id
		})
		response.raise_for_status()

		return cart_item_to_delete

	async def fetch_address(self, params: FetchAddressParams) -> Address:
		addresses_table_id = CoreApiHttpClientConfig.table_mappings['addresses']['id']
		end_point = f"/api/v2/tables/{addresses_table_id}/records/{params.address_id}"
		response = await self._http_client.get(end_point)
		response.raise_for_status()

		unparsed_address = response.json()

		# Parse the Address object into a Address object
		def parse_address(unparsed_address: Dict[str, Any]) -> Address:
			return Address(
				id=unparsed_address.get('Id'),
				user_id=unparsed_address.get('Users_id'),
				address_line=unparsed_address.get('AddressLine'),
				city=unparsed_address.get('City'),
				state=unparsed_address.get('State'),
				country=unparsed_address.get('Country'),
				postal_code=unparsed_address.get('PostalCode'),
				created_at=unparsed_address.get('CreatedAt'), 
				updated_at=unparsed_address.get('UpdatedAt'),	
			)

		address = parse_address(unparsed_address)
		return address
		
	async def create_address(self, params: CreateAddressParams) -> Address:
		addresses_table_id = CoreApiHttpClientConfig.table_mappings['addresses']['id']
		end_point = f"/api/v2/tables/{addresses_table_id}/records"
		response = await self._http_client.post(end_point, json={
			"Title": params.address_line,
			"Users_id": params.user_id,
			"AddressLine": params.address_line,	
			"City": params.city,
			"State": params.state,
			"Country": params.country,
			"PostalCode": params.postal_code,
		})
		response.raise_for_status()

		address_id = response.json().get('Id')
		address = await self.fetch_address(params=FetchAddressParams(address_id=address_id))

		return address

	async def fetch_cart_item(self, params: FetchCartItemParams) -> CartItem:
		cart_items_table_id = CoreApiHttpClientConfig.table_mappings['cart_items']['id']
		end_point = f"/api/v2/tables/{cart_items_table_id}/records/{params.cart_item_id}"
		response = await self._http_client.get(end_point)
		response.raise_for_status()

		unparsed_cart_item = response.json()

		# Parse the Cart_item object into a Cart_item object
		def parse_cart_item(unparsed_cart_item: Dict[str, Any]) -> CartItem:
			return CartItem(
				id=unparsed_cart_item.get('Id'),
				user_id=unparsed_cart_item.get('Users_id'),
				product_variant_id=unparsed_cart_item.get('ProductVariants_id'),	
				quantity=unparsed_cart_item.get('Quantity'),
				created_at=unparsed_cart_item.get('CreatedAt'), 
				updated_at=unparsed_cart_item.get('UpdatedAt'),	
			)

		cart_item = parse_cart_item(unparsed_cart_item)
		return cart_item

	async def create_cart_item(self, params: CreateCartItemParams) -> CartItem:
		cart_items_table_id = CoreApiHttpClientConfig.table_mappings['cart_items']['id']
		end_point = f"/api/v2/tables/{cart_items_table_id}/records"
		response = await self._http_client.post(end_point, json={
			"Users_id": params.user_id,
			"ProductVariants_id": params.product_variant_id,
			"Quantity": params.quantity
		})
		response.raise_for_status()

		cart_item_id = response.json().get('Id')
		cart_item = await self.fetch_cart_item(params=FetchCartItemParams(cart_item_id=cart_item_id))

		return cart_item
	
	async def fetch_order_item(self, params: FetchOrderItemParams) -> OrderItem:
		order_items_table_id = CoreApiHttpClientConfig.table_mappings['order_items']['id']
		end_point = f"/api/v2/tables/{order_items_table_id}/records/{params.order_item_id}"
		response = await self._http_client.get(end_point)
		response.raise_for_status()

		unparsed_order_item = response.json()

		# Parse the Order_item object into a Order_item object
		def parse_order_item(unparsed_order_item: Dict[str, Any]) -> OrderItem:
			return OrderItem(
				id=unparsed_order_item.get('Id'),
				order_id=unparsed_order_item.get('Orders_id'),
				product_variant_id=unparsed_order_item.get('ProductVariants_id'),	
				total_price=unparsed_order_item.get('TotalPrice'),
				quantity=unparsed_order_item.get('Quantity'),
				created_at=unparsed_order_item.get('CreatedAt'), 
				updated_at=unparsed_order_item.get('UpdatedAt'),	
			)

		order_item = parse_order_item(unparsed_order_item)
		return order_item
		
	async def create_order_item(self, params: CreateOrderItemParams) -> OrderItem:
		order_items_table_id = CoreApiHttpClientConfig.table_mappings['order_items']['id']
		end_point = f"/api/v2/tables/{order_items_table_id}/records"

		product_variant = await self.fetch_product_variant(
			params=FetchProductVariantParams(
				product_variant_id=params.product_variant_id
			)
		)

		response = await self._http_client.post(end_point, json={
			"Orders_id": params.order_id,
			"ProductVariants_id": params.product_variant_id,
			"Quantity": params.quantity,
			"TotalPrice": product_variant.price * params.quantity
		})
		response.raise_for_status()

		order_item_id = response.json().get('Id')
		order_item = await self.fetch_order_item(
			params=FetchOrderItemParams(
				order_item_id=order_item_id
			)
		)

		return order_item
	
	async def fetch_order(self, params: FetchOrderParams) -> Order:
		orders_table_id = CoreApiHttpClientConfig.table_mappings['orders']['id']
		end_point = f"/api/v2/tables/{orders_table_id}/records/{params.order_id}"
		response = await self._http_client.get(end_point)
		response.raise_for_status()

		unparsed_order = response.json()

		# Parse the Order object into a Order object
		def parse_order(unparsed_order: Dict[str, Any]) -> Order:
			return Order(
				id=unparsed_order.get('Id'),
				user_id=unparsed_order.get('Users_id'),
				address_id=unparsed_order.get('Addresses_id'),
				status=unparsed_order.get('Status'),
				total_price=unparsed_order.get('TotalPrice'),
				created_at=unparsed_order.get('CreatedAt'), 
				updated_at=unparsed_order.get('UpdatedAt'),	
			)

		order = parse_order(unparsed_order)
		return order

	async def create_order(self, params: CreateOrderParams) -> Order:
		orders_table_id = CoreApiHttpClientConfig.table_mappings['orders']['id']
		end_point = f"/api/v2/tables/{orders_table_id}/records"
		response = await self._http_client.post(end_point, json={
			"Users_id": params.user_id,
			"Addresses_id": params.address_id,
			"TotalPrice": params.total_price,
		})
		response.raise_for_status()

		order_id = response.json().get('Id')
		order = await self.fetch_order(params=FetchOrderParams(order_id=order_id))

		return order
	
	async def update_order(self, params: UpdateOrderParams) -> Order:
		orders_table_id = CoreApiHttpClientConfig.table_mappings['orders']['id']
		end_point = f"/api/v2/tables/{orders_table_id}/records"
		response = await self._http_client.patch(end_point, json={
			"Id": params.order_id,
			"Status": params.status,
			"TotalPrice": params.total_price,
		})
		response.raise_for_status()

		order_id = response.json().get('Id')
		order = await self.fetch_order(params=FetchOrderParams(order_id=order_id))

		return order


#### Testing the functions

In [147]:
core_api_http_client: CoreApiHttpClientABC = CoreApiHttpClient()

Product related

In [148]:
await core_api_http_client.fetch_product_categories()

[ProductCategory(id=1, name='Drum Handling Equipment', description='Drum handling equipment are used to safely and efficiently move, lift and pour to discharge drum content. They are the solutions to address every drum handling issues as unsafe handling can result in cost of content or any unnecessary injuries or damages. \n\nDrum handling equipment are particularly used in factories which involve drums in their daily operation food & beverage, & gas, pharmaceutical; chemical; printing industries & etc. They are also used in logistic centres which are involved in distributing drums.', product_count=4, created_at=datetime.datetime(2025, 1, 30, 16, 4, 33, tzinfo=TzInfo(UTC)), updated_at=datetime.datetime(2025, 1, 30, 16, 10, 1, tzinfo=TzInfo(UTC)), products=None),
 ProductCategory(id=2, name='Stacker', description='Stackers are economical alternative to a forklift, commonly used to transfer load onto low or mid-level racking; making the tasks quicker and safer. Because they are smaller i

In [149]:
await core_api_http_client.fetch_product_variant(params=FetchProductVariantParams(product_variant_id=1))

ProductVariant(id=1, name='Auto N-1', price=500.0, stock=10, product_id=1, created_at=datetime.datetime(2025, 1, 30, 14, 39, 1, tzinfo=TzInfo(UTC)), updated_at=datetime.datetime(2025, 2, 4, 14, 14, 30, tzinfo=TzInfo(UTC)), product=None, cart_items=None, order_items=None)

In [150]:
await core_api_http_client.fetch_products_by_product_category(params=FetchProductsByProductCategoryParams(product_category_id=1))

[Product(id=3, name='Drum Handler', description='Strong gripping mechanism.\nDrum is safely supported while tilting.\nLight and smooth tilting even when drum is full loaded.\nCompact and maneuverable.\nTough; reliable and productive.', product_variant_count=1, product_category_id=1, created_at=datetime.datetime(2025, 1, 30, 14, 58, 26, tzinfo=TzInfo(UTC)), updated_at=datetime.datetime(2025, 1, 30, 15, 2, 57, tzinfo=TzInfo(UTC)), product_variants=None, product_category=None),
 Product(id=4, name='Hydraulic Drum Porter', description='Strong gripping mechanism.  \nLightweight and effortless maneuvering.\nEasy to operate.\nLow maintenance cost.\nReliable; safe and efficient.', product_variant_count=4, product_category_id=1, created_at=datetime.datetime(2025, 1, 30, 15, 3, 3, tzinfo=TzInfo(UTC)), updated_at=datetime.datetime(2025, 1, 30, 15, 7, 30, tzinfo=TzInfo(UTC)), product_variants=None, product_category=None),
 Product(id=1, name='Drum Gripper', description="Auto handling - entire oper

In [151]:
await core_api_http_client.fetch_product(params=FetchProductParams(product_id=1))

Product(id=1, name='Drum Gripper', description="Auto handling - entire operation s controlled from the driver's seat.\nStrong gripping mechanism, drum is handled safely and efficiently (for N series)\nCradle belt protects drum from dents or scratches (for N Series)\nDrum is grasped securely by the rim of the drum (for U Series)\nJaw can be adjusted manually to fit the diameter of the drum (for U Series)\nEasy fast and safe", product_variant_count=4, product_category_id=1, created_at=datetime.datetime(2025, 1, 30, 14, 13, 1, tzinfo=TzInfo(UTC)), updated_at=datetime.datetime(2025, 1, 30, 14, 58, 43, tzinfo=TzInfo(UTC)), product_variants=None, product_category=None)

In [152]:
await core_api_http_client.fetch_product_variants_by_product(params=FetchProductVariantsByProductParams(product_id=1))

[ProductVariant(id=1, name='Auto N-1', price=500.0, stock=10, product_id=1, created_at=datetime.datetime(2025, 1, 30, 14, 39, 1, tzinfo=TzInfo(UTC)), updated_at=datetime.datetime(2025, 2, 4, 14, 14, 30, tzinfo=TzInfo(UTC)), product=None, cart_items=None, order_items=None),
 ProductVariant(id=3, name='Auto U-1', price=500.0, stock=10, product_id=1, created_at=datetime.datetime(2025, 1, 30, 14, 41, 3, tzinfo=TzInfo(UTC)), updated_at=datetime.datetime(2025, 2, 4, 16, 22, 10, tzinfo=TzInfo(UTC)), product=None, cart_items=None, order_items=None),
 ProductVariant(id=4, name='Auto U-2', price=500.0, stock=10, product_id=1, created_at=datetime.datetime(2025, 1, 30, 14, 41, 28, tzinfo=TzInfo(UTC)), updated_at=datetime.datetime(2025, 1, 30, 14, 59, 12, tzinfo=TzInfo(UTC)), product=None, cart_items=None, order_items=None),
 ProductVariant(id=2, name='Auto N-2', price=500.0, stock=10, product_id=1, created_at=datetime.datetime(2025, 1, 30, 14, 40, 12, tzinfo=TzInfo(UTC)), updated_at=datetime.datet

User related

In [153]:
await core_api_http_client.fetch_user_by_email(params=FetchUserByEmailParams(email='david@gmail.com'))

User(id=41, name='david delacroz', email='david@gmail.com', phone_number='+639292557199', address_count=16, cart_item_count=0, created_at=datetime.datetime(2025, 2, 4, 12, 44, 19, tzinfo=TzInfo(UTC)), updated_at=datetime.datetime(2025, 2, 6, 11, 17, 53, tzinfo=TzInfo(UTC)), addresses=None, cart_items=None, orders=None)

In [154]:
await core_api_http_client.fetch_addresses_by_user(params=FetchAddressesByUserParams(user_id=41))

[Address(id=35, user_id=41, address_line='45 test address line', city='Test city', state='Test state', postal_code='4002', country='Test country', created_at=datetime.datetime(2025, 2, 6, 4, 57, 32, tzinfo=TzInfo(UTC)), updated_at=None, user=None, orders=None),
 Address(id=36, user_id=41, address_line='45 test address line', city='Test city', state='Test state', postal_code='4002', country='Test country', created_at=datetime.datetime(2025, 2, 6, 5, 0, 46, tzinfo=TzInfo(UTC)), updated_at=None, user=None, orders=None),
 Address(id=37, user_id=41, address_line='45 test address line', city='Test city', state='Test state', postal_code='4002', country='Test country', created_at=datetime.datetime(2025, 2, 6, 8, 3, 52, tzinfo=TzInfo(UTC)), updated_at=None, user=None, orders=None),
 Address(id=38, user_id=41, address_line='45 test address line', city='Test city', state='Test state', postal_code='4002', country='Test country', created_at=datetime.datetime(2025, 2, 6, 8, 6, 9, tzinfo=TzInfo(UTC))

In [155]:
await core_api_http_client.fetch_address(params=FetchAddressParams(address_id=27))

Address(id=27, user_id=41, address_line='45 Maple Lane, Queens, New York', city='Queens', state='New York', postal_code='4002', country='USA', created_at=datetime.datetime(2025, 2, 4, 16, 12, 10, tzinfo=TzInfo(UTC)), updated_at=datetime.datetime(2025, 2, 4, 16, 14, 45, tzinfo=TzInfo(UTC)), user=None, orders=None)

In [156]:
await core_api_http_client.create_address(
	params=CreateAddressParams(
		user_id=41, 
		address_line='45 test address line', 
		city='Test city', 
		state='Test state', 
		postal_code='4002', 
		country='Test country'
	)
)	

Address(id=51, user_id=41, address_line='45 test address line', city='Test city', state='Test state', postal_code='4002', country='Test country', created_at=datetime.datetime(2025, 2, 6, 11, 50, 15, tzinfo=TzInfo(UTC)), updated_at=None, user=None, orders=None)

Order related

In [157]:
await core_api_http_client.fetch_cart_items_by_user(params=FetchCartItemsByUserParams(user_id=44))

[CartItem(id=60, quantity=2, user_id=44, product_variant_id=19, created_at=datetime.datetime(2025, 2, 5, 9, 21, 18, tzinfo=TzInfo(UTC)), updated_at=None, user=None, product_variant=None),
 CartItem(id=61, quantity=2, user_id=44, product_variant_id=19, created_at=datetime.datetime(2025, 2, 6, 5, 4, 15, tzinfo=TzInfo(UTC)), updated_at=None, user=None, product_variant=None),
 CartItem(id=62, quantity=2, user_id=44, product_variant_id=19, created_at=datetime.datetime(2025, 2, 6, 8, 3, 52, tzinfo=TzInfo(UTC)), updated_at=None, user=None, product_variant=None),
 CartItem(id=63, quantity=2, user_id=44, product_variant_id=19, created_at=datetime.datetime(2025, 2, 6, 8, 6, 9, tzinfo=TzInfo(UTC)), updated_at=None, user=None, product_variant=None),
 CartItem(id=64, quantity=2, user_id=44, product_variant_id=19, created_at=datetime.datetime(2025, 2, 6, 8, 11, 40, tzinfo=TzInfo(UTC)), updated_at=None, user=None, product_variant=None),
 CartItem(id=65, quantity=2, user_id=44, product_variant_id=19, 

In [158]:
await core_api_http_client.fetch_cart_item(params=FetchCartItemParams(cart_item_id=60))

CartItem(id=60, quantity=2, user_id=44, product_variant_id=19, created_at=datetime.datetime(2025, 2, 5, 9, 21, 18, tzinfo=TzInfo(UTC)), updated_at=None, user=None, product_variant=None)

In [159]:
await core_api_http_client.create_cart_item(
	params=CreateCartItemParams(
		user_id=44,
		product_variant_id=19,
		quantity=2
	)
)

CartItem(id=76, quantity=2, user_id=44, product_variant_id=19, created_at=datetime.datetime(2025, 2, 6, 11, 50, 15, tzinfo=TzInfo(UTC)), updated_at=None, user=None, product_variant=None)

In [160]:
await core_api_http_client.fetch_order_item(params=FetchOrderItemParams(order_item_id=34))

OrderItem(id=34, order_id=28, product_variant_id=18, quantity=1, total_price=500.0, created_at=datetime.datetime(2025, 2, 5, 3, 8, 41, tzinfo=TzInfo(UTC)), updated_at=datetime.datetime(2025, 2, 6, 6, 21, 23, tzinfo=TzInfo(UTC)), order=None, product_variant=None)

In [161]:
await core_api_http_client.create_order_item(
	params=CreateOrderItemParams(
		order_id=28,
		product_variant_id=18,
		quantity=1
	)
)

OrderItem(id=101, order_id=28, product_variant_id=18, quantity=1, total_price=500.0, created_at=datetime.datetime(2025, 2, 6, 11, 50, 16, tzinfo=TzInfo(UTC)), updated_at=None, order=None, product_variant=None)

In [162]:
await core_api_http_client.fetch_order(params=FetchOrderParams(order_id=28))

Order(id=28, user_id=41, address_id=27, status='PENDING', total_price=500.0, created_at=datetime.datetime(2025, 2, 5, 3, 8, 41, tzinfo=TzInfo(UTC)), updated_at=None, user=None, address=None, order_items=None)

In [163]:
await core_api_http_client.create_order(
	params=CreateOrderParams(
		user_id=41,
		address_id=27,
		total_price=500.0
	)
)

Order(id=54, user_id=41, address_id=27, status='PENDING', total_price=500.0, created_at=datetime.datetime(2025, 2, 6, 11, 50, 16, tzinfo=TzInfo(UTC)), updated_at=None, user=None, address=None, order_items=None)

In [164]:
await core_api_http_client.update_order(
	params=UpdateOrderParams(
		order_id=29,
		status=OrderStatus.TO_SHIP,
		total_price=500
	)
)

Order(id=29, user_id=41, address_id=27, status='TO_SHIP', total_price=500.0, created_at=datetime.datetime(2025, 2, 6, 6, 37, 59, tzinfo=TzInfo(UTC)), updated_at=datetime.datetime(2025, 2, 6, 11, 50, 16, tzinfo=TzInfo(UTC)), user=None, address=None, order_items=None)

### Repository layer

Product category repository

In [165]:
@container.abstract
class ProductCategoryRepositoryABC(ABC):
	@abstractmethod
	async def fetch_all(self) -> List[ProductCategory]:
		pass

In [166]:
@container.register
class ProductCategoryRepository(ProductCategoryRepositoryABC):
	@container.autowire
	def __init__(self, core_api_http_client: CoreApiHttpClientABC):
		self._core_api_http_client = core_api_http_client

	async def fetch_all(self) -> List[ProductCategory]:
		return await self._core_api_http_client.fetch_product_categories()

Product repository

In [167]:
class ProductRepositoryFetchParams(BaseModel):
	product_id: int

class ProductRepositoryFetchByCategoryParams(BaseModel):
	product_category_id: int	

@container.abstract
class ProductRepositoryABC(ABC):
	@abstractmethod
	async def fetch(self, params: ProductRepositoryFetchParams) -> Product:
		pass

	@abstractmethod
	async def fetch_by_product_category(self, params: ProductRepositoryFetchByCategoryParams) -> List[Product]:
		pass

@container.register
class ProductRepository(ProductRepositoryABC):
	@container.autowire
	def __init__(self, core_api_http_client: CoreApiHttpClientABC):
		self._core_api_http_client = core_api_http_client

	async def fetch(self, params: ProductRepositoryFetchParams) -> Product:
		return await self._core_api_http_client.fetch_product(
			params=FetchProductParams(
				product_id=params.product_id
			)
		)

	async def fetch_by_product_category(self, params: ProductRepositoryFetchByCategoryParams) -> List[Product]:
		return await self._core_api_http_client.fetch_products_by_product_category(
			params=FetchProductsByProductCategoryParams(
				product_category_id = params.product_category_id
			)
		)

Product variant repository

In [168]:
class ProductVariantRepositoryFetchParams(BaseModel):
	product_variant_id: int

class ProductVariantRepositoryFetchByProductParams(BaseModel):
	product_id: int

@container.abstract
class ProductVariantRepositoryABC(ABC):
	@abstractmethod
	async def fetch(self, params: ProductVariantRepositoryFetchParams) -> ProductVariant:
		pass

	@abstractmethod
	async def fetch_by_product(self, params: ProductVariantRepositoryFetchByProductParams) -> List[ProductVariant]:
		pass

@container.register
class ProductVariantRepository(ProductVariantRepositoryABC):
	@container.autowire
	def __init__(self, core_api_http_client: CoreApiHttpClientABC):
		self._core_api_http_client = core_api_http_client

	async def fetch(self, params: ProductVariantRepositoryFetchParams) -> ProductVariant:
		return await self._core_api_http_client.fetch_product_variant(
			params=FetchProductVariantParams(
				product_variant_id=params.product_variant_id
			)
		)

	async def fetch_by_product(self, params: ProductVariantRepositoryFetchByProductParams) -> List[ProductVariant]:
		return await self._core_api_http_client.fetch_product_variants_by_product(
			params=FetchProductVariantsByProductParams(
				product_id=params.product_id
			)
		)

User repository

In [169]:
class UserRepositoryFetchParams(BaseModel):
	user_id: int

class UserRepositoryFetchByEmailParams(BaseModel):
	email: str

@container.abstract	
class UserRepositoryABC(ABC):
	@abstractmethod
	async def fetch(self, params: UserRepositoryFetchParams) -> User:
		pass

	@abstractmethod
	async def fetch_by_email(self, params: UserRepositoryFetchByEmailParams) -> Optional[User]:
		pass

@container.register	
class UserRepository(UserRepositoryABC):
	@container.autowire
	def __init__(self, core_api_http_client: CoreApiHttpClientABC):
		self._core_api_http_client = core_api_http_client

	async def fetch(self, params: UserRepositoryFetchParams) -> User:
		return await self._core_api_http_client.fetch_user(
			params=FetchUserParams(
				user_id=params.user_id
			)
		)
	
	async def fetch_by_email(self, params: UserRepositoryFetchByEmailParams) -> Optional[User]:
		return await self._core_api_http_client.fetch_user_by_email(
			params=FetchUserByEmailParams(
				email=params.email
			)
		)

Address repository

In [170]:
class AddressRepositoryFetchParams(BaseModel):
	address_id: int	

class AddressRepositoryCreateParams(BaseModel):
	user_id: int
	address_line: str
	city: str
	state: str
	country: str
	postal_code: str

@container.abstract
class AddressRepositoryABC(ABC):
	@abstractmethod
	async def fetch(self, params: AddressRepositoryFetchParams) -> Address:
		pass

	@abstractmethod
	async def create(self, params: AddressRepositoryCreateParams) -> Address:
		pass

@container.register
class AddressRepository(AddressRepositoryABC):
	@container.autowire
	def __init__(self, core_api_http_client: CoreApiHttpClientABC):
		self._core_api_http_client = core_api_http_client

	async def fetch(self, params: AddressRepositoryFetchParams) -> Address:
		return await self._core_api_http_client.fetch_address(
			params=FetchAddressParams(
				address_id=params.address_id
			)
		)

	async def create(self, params: AddressRepositoryCreateParams) -> Address:
		return await self._core_api_http_client.create_address(
			params=CreateAddressParams(
				user_id=params.user_id,
				address_line=params.address_line,
				city=params.city,
				state=params.state,
				country=params.country,
				postal_code=params.postal_code
			)
		)

Cart item repository

In [171]:
class CartItemRepositoryFetchParams(BaseModel):
	cart_item_id: int

class CartItemRepositoryCreateParams(BaseModel):
	user_id: int
	product_variant_id: int
	quantity: int

class CartItemRepositoryFetchByUserParams(BaseModel):
	user_id: int

class CartItemRepositoryDeleteParams(BaseModel):
	cart_item_id: int

@container.abstract
class CartItemRepositoryABC(ABC):
	@abstractmethod
	async def fetch(self, params: CartItemRepositoryFetchParams) -> CartItem:
		pass

	@abstractmethod
	async def create(self, params: CartItemRepositoryCreateParams) -> CartItem:
		pass

	@abstractmethod
	async def fetch_by_user(self, params: CartItemRepositoryFetchByUserParams) -> List[CartItem]:
		pass

	@abstractmethod
	async def delete(self, params: CartItemRepositoryDeleteParams) -> None:
		pass

@container.register
class CartItemRepository(CartItemRepositoryABC):
	@container.autowire
	def __init__(self, core_api_http_client: CoreApiHttpClientABC):
		self._core_api_http_client = core_api_http_client

	async def fetch(self, params: CartItemRepositoryFetchParams) -> CartItem:
		return await self._core_api_http_client.fetch_cart_item(
			params=FetchCartItemParams(
				cart_item_id=params.cart_item_id
			)
		)

	async def create(self, params: CartItemRepositoryCreateParams) -> CartItem:
		return await self._core_api_http_client.create_cart_item(
			params=CreateCartItemParams(
				user_id=params.user_id,
				product_variant_id=params.product_variant_id,
				quantity=params.quantity
			)
		)
	
	async def fetch_by_user(self, params: CartItemRepositoryFetchByUserParams) -> List[CartItem]:
		return await self._core_api_http_client.fetch_cart_items_by_user(
			params=FetchCartItemsByUserParams(
				user_id=params.user_id
			)
		)
	
	async def delete(self, params: CartItemRepositoryDeleteParams) -> None:
		await self._core_api_http_client.delete_cart_item(
			params=DeleteCartItemParams(
				cart_item_id=params.cart_item_id
			)
		)

Order item repository

In [172]:
class OrderItemRepositoryFetchParams(BaseModel):
	order_item_id: int

class OrderItemRepositoryCreateParams(BaseModel):
	order_id: int
	product_variant_id: int
	quantity: int

@container.abstract
class OrderItemRepositoryABC(ABC):
	@abstractmethod
	async def fetch(self, params: OrderItemRepositoryFetchParams) -> OrderItem:
		pass

	@abstractmethod
	async def create(self, params: OrderItemRepositoryCreateParams) -> OrderItem:
		pass

@container.register
class OrderItemRepository(OrderItemRepositoryABC):
	@container.autowire
	def __init__(self, core_api_http_client: CoreApiHttpClientABC):
		self._core_api_http_client = core_api_http_client

	async def fetch(self, params: OrderItemRepositoryFetchParams) -> OrderItem:
		return await self._core_api_http_client.fetch_order_item(
			params=FetchOrderItemParams(
				order_item_id=params.order_item_id
			)
		)

	async def create(self, params: OrderItemRepositoryCreateParams) -> OrderItem:
		return await self._core_api_http_client.create_order_item(
			params=CreateOrderItemParams(
				order_id=params.order_id,
				product_variant_id=params.product_variant_id,
				quantity=params.quantity
			)
		)

Order repository

In [173]:
class OrderRepositoryFetchParams(BaseModel):
	order_id: int

class OrderRepositoryCreateParams(BaseModel):
	user_id: int
	address_id: int
	total_price: float

@container.abstract
class OrderRepositoryABC(ABC):
	@abstractmethod
	async def fetch(self, params: OrderRepositoryFetchParams) -> Order:
		pass

	@abstractmethod
	async def create(self, params: OrderRepositoryCreateParams) -> Order:
		pass

@container.register
class OrderRepository(OrderRepositoryABC):
	@container.autowire
	def __init__(self, core_api_http_client: CoreApiHttpClientABC):
		self._core_api_http_client = core_api_http_client

	async def fetch(self, params: OrderRepositoryFetchParams) -> Order:
		return await self._core_api_http_client.fetch_order(
			params=FetchOrderParams(
				order_id=params.order_id
			)
		)

	async def create(self, params: OrderRepositoryCreateParams) -> Order:
		return await self._core_api_http_client.create_order(
			params=CreateOrderParams(
				user_id=params.user_id,
				address_id=params.address_id,
				total_price=params.total_price
			)
		)

### Service layer

Product category service

In [174]:
@container.abstract
class ProductCategoryServiceABC(ABC):
	@abstractmethod
	async def fetch_all(self) -> List[ProductCategory]:
		pass

@container.register
class ProductCategoryService(ProductCategoryServiceABC):
	@container.autowire
	def __init__(self, product_category_repository: ProductCategoryRepositoryABC):
		self._product_category_repository = product_category_repository

	async def fetch_all(self) -> List[ProductCategory]:
		return await self._product_category_repository.fetch_all()

Product service

In [175]:
class ProductServiceFetchProductsWithVariantsByProductCategoryParams(BaseModel):
	product_category_id: int

@container.abstract
class ProductServiceABC(ABC):
	@abstractmethod
	async def fetch_products_with_variants_by_product_category(self, params: ProductServiceFetchProductsWithVariantsByProductCategoryParams):
		pass

@container.register
class ProductService(ProductServiceABC):
	@container.autowire
	def __init__(self, product_repository: ProductRepositoryABC, product_variant_repository: ProductVariantRepositoryABC):
		self._product_repository = product_repository
		self._product_variant_repository = product_variant_repository

	async def fetch_products_with_variants_by_product_category(self, params: ProductServiceFetchProductsWithVariantsByProductCategoryParams):
		products_by_product_category = await self._product_repository.fetch_by_product_category(
			params=ProductRepositoryFetchByCategoryParams(
				product_category_id=params.product_category_id
			)
		)

		async def fetch_product_variants(product: Product) -> Product:
			product.product_variants = await self._product_variant_repository.fetch_by_product(
				params=ProductVariantRepositoryFetchByProductParams(
					product_id=product.id
				)
			)
			return product
		
		# Process in batch to not oeverwhelm event loop
		def chunk_list(list_to_chunk, batch_size):
			for i in range(0, len(list_to_chunk), batch_size):
				yield list_to_chunk[i:i + batch_size]


		batch_size: int = 10
		products_with_product_variants: List[Product] = []
		for chunk in chunk_list(products_by_product_category, batch_size):
			batch_results = await asyncio.gather(
				*[fetch_product_variants(product) for product in chunk]
			)

			products_with_product_variants.extend(batch_results)

		return products_with_product_variants

		

Address repository

In [176]:
class AddressRepositoryFetchAddressesByUserParams(BaseModel):
	user_id: int

class AddressRepositoryFetchParams(BaseModel):
	address_id: int

@container.abstract
class AddressRepositoryABC(ABC):
	@abstractmethod
	async def fetch_addresses_by_user(self, params: AddressRepositoryFetchAddressesByUserParams) -> List[Address]:
		pass

	@abstractmethod
	async def fetch(self, params: AddressRepositoryFetchParams) -> Address:
		pass

@container.register
class AddressRepository(AddressRepositoryABC):
	@container.autowire
	def __init__(self, core_api_http_client: CoreApiHttpClientABC):
		self._core_api_http_client = core_api_http_client

	async def fetch_addresses_by_user(self, params: AddressRepositoryFetchAddressesByUserParams) -> List[Address]:
		return await self._core_api_http_client.fetch_addresses_by_user(
			params=FetchAddressesByUserParams(
				user_id=params.user_id
			)
		)

	async def fetch(self, params: AddressRepositoryFetchParams) -> Address:
		return await self._core_api_http_client.fetch_address(
			params=FetchAddressParams(
				address_id=params.address_id
			)	
		)

User service

In [177]:
class UserServiceFetchWithAddressesParams(BaseModel):
	user_id: int

class UserServiceFetchWithAddressesByEmailParams(BaseModel):
	email: str

@container.abstract
class UserServiceABC(ABC):
	@abstractmethod
	async def fetch_with_addresses(self, params: UserServiceFetchWithAddressesParams):
		pass

	@abstractmethod
	async def fetch_with_addresses_by_email(self, params: UserServiceFetchWithAddressesByEmailParams):
		pass

@container.register
class UserService(UserServiceABC):
	@container.autowire
	def __init__(self, user_repository: UserRepositoryABC, address_repository: AddressRepositoryABC):
		self._user_repository = user_repository
		self._address_repository = address_repository

	async def fetch_with_addresses(self, params: UserServiceFetchWithAddressesParams):
		user = await self._user_repository.fetch(
			params=UserRepositoryFetchParams(
				user_id=params.user_id
			)
		)

		user.addresses = await self._address_repository.fetch_addresses_by_user(
			params=AddressRepositoryFetchAddressesByUserParams(
				user_id=user.id
			)
		)

		return user

	async def fetch_with_addresses_by_email(self, params: UserServiceFetchWithAddressesByEmailParams):
		user = await self._user_repository.fetch_by_email(
			params=UserRepositoryFetchByEmailParams(
				email=params.email
			)
		)

		user.addresses = await self._address_repository.fetch_addresses_by_user(
			params=AddressRepositoryFetchAddressesByUserParams(
				user_id=user.id
			)
		)

		return user

Cart items service

In [178]:
class CartItemServiceFetchByUserParams(BaseModel):
	user_id: int

class CartItemServiceCreateParams(BaseModel):
	user_id: int
	product_variant_id: int
	quantity: int

@container.abstract
class CartItemServiceABC(ABC):
	@abstractmethod
	async def fetch_by_user(self, params: CartItemServiceFetchByUserParams):
		pass

	@abstractmethod
	async def create(self, params: CartItemServiceCreateParams):
		pass

@container.register
class CartItemService(CartItemServiceABC):
	@container.autowire
	def __init__(self, cart_item_repository: CartItemRepositoryABC):
		self._cart_item_repository = cart_item_repository

	async def fetch_by_user(self, params: CartItemServiceFetchByUserParams):
		return await self._cart_item_repository.fetch_by_user(
			params=CartItemRepositoryFetchByUserParams(
				user_id=params.user_id
			)
		)

	async def create(self, params: CartItemServiceCreateParams):
		return await self._cart_item_repository.create(
			params=CartItemRepositoryCreateParams(
				user_id=params.user_id,
				product_variant_id=params.product_variant_id,
				quantity=params.quantity
			)
	)

Order service

In [179]:
class OrderServiceCreateParams(BaseModel):
	user_id: int
	address_id: int

@container.abstract
class OrderServiceABC(ABC):
	@abstractmethod
	async def create(self, params: OrderServiceCreateParams):
		pass

@container.register
class OrderService(OrderServiceABC):
	@container.autowire
	def __init__(
		self, 
		order_repository: OrderRepositoryABC, 
		order_item_repository: OrderItemRepositoryABC,
		cart_item_repository: CartItemRepositoryABC,
		product_variant_repository: ProductVariantRepositoryABC
	):
		self._order_repository = order_repository
		self._order_item_repository = order_item_repository
		self._cart_item_repository = cart_item_repository
		self._product_variant_repository = product_variant_repository

	async def create(self, params: OrderServiceCreateParams):
		cart_items: List[CartItem] = await self._cart_item_repository.fetch_by_user(
			params=CartItemRepositoryFetchByUserParams(
				user_id=params.user_id
			)
		)

		batch_size: int = 10
		def chunk_list(list_to_chunk: list, batch_size: int = batch_size):
			for i in range(0, len(list_to_chunk), batch_size):
				yield list_to_chunk[i:i + batch_size]

		# 1. Populate cart item's product variant
		async def populate_cart_item_product_variant(cart_item: CartItem):
			cart_item.product_variant = await self._product_variant_repository.fetch(
				params=ProductVariantRepositoryFetchParams(
					product_variant_id=cart_item.product_variant_id
				)
			)

			return cart_item
		
		processed_cart_items: List[CartItem] = []
		for chunk in chunk_list(list_to_chunk=cart_items):
			batch_results = await asyncio.gather(
				*[populate_cart_item_product_variant(cart_item) for cart_item in chunk]
			)

			processed_cart_items.extend(batch_results)
		
		cart_items = processed_cart_items

		# 2. Compute total price 
		with concurrent.futures.ThreadPoolExecutor() as executor:
			cart_items_total_prices = list(executor.map(lambda x: x.product_variant.price * x.quantity, cart_items))
		cart_item_total_price = sum(cart_items_total_prices)


		# 3. Create order record
		order = await self._order_repository.create(
			params=OrderRepositoryCreateParams(
				user_id=params.user_id,
				address_id=params.address_id,
				total_price=cart_item_total_price
			)
		)


		# 4. Add to order items and remove from cart
		async def create_order_item_transaction(cart_item: CartItem):
			order_item = await self._order_item_repository.create(
				params=OrderItemRepositoryCreateParams(
					order_id=order.id,
					product_variant_id=cart_item.product_variant_id,
					quantity=cart_item.quantity
				)
			)

			await self._cart_item_repository.delete(
				params=CartItemRepositoryDeleteParams(
					cart_item_id=cart_item.id
				)
			)

			return order_item
		
		batch_processed_order_items: List[OrderItem] = []
		for chunk in chunk_list(list_to_chunk=cart_items):
			batch_results = await asyncio.gather(
				*[create_order_item_transaction(cart_item) for cart_item in chunk]
			)

			batch_processed_order_items.extend(batch_results)


		# 5. Fetch order with order items
		order = await self._order_repository.fetch(
			params=OrderRepositoryFetchParams(
				order_id = order.id
			)
		)

		order.order_items = batch_processed_order_items
		
		return order

In [180]:
await container.get(OrderServiceABC).create(
	params=OrderServiceCreateParams(
		user_id=44,
		address_id=48
	)
)

Order(id=55, user_id=44, address_id=48, status='PENDING', total_price=17000.0, created_at=datetime.datetime(2025, 2, 6, 11, 50, 17, tzinfo=TzInfo(UTC)), updated_at=None, user=None, address=None, order_items=[OrderItem(id=102, order_id=55, product_variant_id=19, quantity=2, total_price=1000.0, created_at=datetime.datetime(2025, 2, 6, 11, 50, 17, tzinfo=TzInfo(UTC)), updated_at=None, order=None, product_variant=None), OrderItem(id=103, order_id=55, product_variant_id=19, quantity=2, total_price=1000.0, created_at=datetime.datetime(2025, 2, 6, 11, 50, 17, tzinfo=TzInfo(UTC)), updated_at=None, order=None, product_variant=None), OrderItem(id=110, order_id=55, product_variant_id=19, quantity=2, total_price=1000.0, created_at=datetime.datetime(2025, 2, 6, 11, 50, 17, tzinfo=TzInfo(UTC)), updated_at=None, order=None, product_variant=None), OrderItem(id=107, order_id=55, product_variant_id=19, quantity=2, total_price=1000.0, created_at=datetime.datetime(2025, 2, 6, 11, 50, 17, tzinfo=TzInfo(UTC